In [1]:
# ============================================================
# CELL 1 — Config + Cross-Lakehouse Connectivity Test
# Gold Transform: silver_globalwatch → gold_globalwatch
# Star Schema: fact_readings + 4 dimensions
#
# SCD Strategy:
#   dim_date      → Type 0 (static spine, never changes)
#   dim_pollutant → Type 0 (static reference, never changes)
#   dim_country   → Type 1 (overwrite — no history needed)
#   dim_station   → Type 2 (Delta MERGE — track name/city changes)
#
# Spark Optimizations applied in this notebook:
#   - AQE: dynamic partition coalescing + skew join handling
#   - Broadcast join: small dims (<50MB) auto-broadcast to executors
#   - V-Order: Gold tables written with V-Order for Direct Lake
#   - Z-Order: fact_readings clustered on location_id + reading_ts
# ============================================================

# --- Spark Optimization Config ---
# AQE: Spark dynamically optimizes execution plan at runtime
# Eliminates need to manually tune spark.sql.shuffle.partitions
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Coalesce: merges small shuffle partitions automatically
# Prevents 200 tiny output files on small datasets
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Skew join: splits oversized partitions (e.g. Beijing station data)
# Prevents single executor OOM on high-volume city stations
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Broadcast threshold: any table under 50MB is auto-broadcast
# dim_country (~9 rows), dim_pollutant (5 rows) will always broadcast
# Eliminates shuffle entirely for dimension joins
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))

# --- Standard Imports ---
from pyspark.sql import functions as F       # PySpark column functions
from pyspark.sql.types import *              # Schema type definitions
from pyspark.sql.window import Window        # one-row-per-station dedupe
from delta.tables import DeltaTable          # Delta MERGE API for SCD2
from datetime import datetime, timezone      # UTC timestamp handling

# --- Gold DB Context ---
# Fabric encodes workspace+lakehouse into an internal DB name
# current_database() returns the exact string to use in all table refs
GOLD_DB = spark.sql("SELECT current_database()").collect()[0][0]

# --- Cross-Lakehouse Read Test ---
# Silver lakehouse is a separate OneLake item
# Fabric allows cross-lakehouse reads via 3-part table reference:
# <lakehouse_name>.dbo.<table_name>
# This works because all lakehouses share the same OneLake storage
df_silver = spark.read \
    .format("delta") \
    .table("silver_globalwatch.dbo.silver_readings")

silver_count = df_silver.count()

print(f"Config loaded ✅")
print(f"Gold DB   : {GOLD_DB}")
print(f"Silver rows available: {silver_count}")
print(f"Silver columns: {df_silver.columns}")

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 3, Finished, Available, Finished, False)

Config loaded ✅
Gold DB   : chimcobldhq2aprcdth62r3nc5q66q1dchinc9b7dtm68nr7dhnm4obcetgn8or84li64ro
Silver rows available: 344
Silver columns: ['location_id', 'location_name', 'city', 'country_code', 'country_name', 'latitude', 'longitude', 'parameter', 'value', 'unit', 'aqi_category', 'reading_ts', 'reading_date', 'reading_hour', 'year_month', 'is_recent', 'ingestion_ts', 'silver_processed_ts', 'source_system']


In [2]:
# ============================================================
# CELL 2 — dim_date (Static Date Spine)
# Type 0 SCD — never changes once created
# Pre-generated: 2015-01-01 to 2030-12-31 = 5,844 rows
#
# Why pre-generate vs compute on the fly?
#   - Power BI time intelligence functions (YTD, MTD, QTD)
#     require a complete contiguous date table
#   - Avoids recomputing date attributes on every query
#   - Direct Lake reads dim_date as a pre-sorted Delta file
#     — extremely fast for calendar slicers in Power BI
#
# Interview note: "We pre-generate the date spine once and
# write it as a static Delta table. Power BI's time
# intelligence requires a complete calendar — no gaps."
# ============================================================

from pyspark.sql.types import IntegerType, StringType, DateType, BooleanType

# Generate date spine using Spark SQL sequence function
# sequence(start, end, step) generates an array of dates
# explode() converts array into individual rows
# This avoids a Python loop — pure Spark, fully distributed
df_dates = spark.sql("""
    SELECT
        explode(sequence(
            to_date('2015-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )) AS full_date
""") \
.withColumn(
    # Surrogate key: integer YYYYMMDD format
    # e.g. 2026-08-08 → 20260808
    # Integer keys are faster for joins than date types
    "date_key",
    F.date_format("full_date", "yyyyMMdd").cast(IntegerType())
) \
.withColumn(
    # Calendar year: 2015, 2016 ... 2030
    "year",
    F.year("full_date")
) \
.withColumn(
    # Calendar month number: 1–12
    "month",
    F.month("full_date")
) \
.withColumn(
    # Month display name: January, February ...
    "month_name",
    F.date_format("full_date", "MMMM")
) \
.withColumn(
    # Fiscal quarter: 1–4
    "quarter",
    F.quarter("full_date")
) \
.withColumn(
    # Day of week: 1=Sunday, 7=Saturday (Spark convention)
    "day_of_week",
    F.dayofweek("full_date")
) \
.withColumn(
    # Day display name: Monday, Tuesday ...
    "day_name",
    F.date_format("full_date", "EEEE")
) \
.withColumn(
    # Weekend flag: True for Saturday (7) and Sunday (1)
    # Used in Power BI to filter weekday vs weekend trends
    "is_weekend",
    F.dayofweek("full_date").isin([1, 7])
) \
.withColumn(
    # Year-month string: used for partitioning and slicers
    # e.g. "2026-08"
    "year_month",
    F.date_format("full_date", "yyyy-MM")
)

# --- Write dim_date ---
# mode=overwrite: safe for static tables — same data every run
# No partitioning needed — 5,844 rows fits in a single file
# Direct Lake will scan the whole table for calendar slicers anyway
df_dates.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dim_date")

date_count = df_dates.count()
print(f"dim_date: {date_count} rows ✅")
print(f"Date range: 2015-01-01 → 2030-12-31")
print(f"\nSample rows:")
df_dates.show(3)

# Sanity check: should be exactly 5,844 rows (16 years including leap years)
assert date_count == 5844, f"Expected 5844 rows, got {date_count}"
print(f"Row count assertion passed ✅")

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 5, Finished, Available, Finished, False)

dim_date: 5844 rows ✅
Date range: 2015-01-01 → 2030-12-31

Sample rows:
+----------+--------+----+-----+----------+-------+-----------+--------+----------+----------+
| full_date|date_key|year|month|month_name|quarter|day_of_week|day_name|is_weekend|year_month|
+----------+--------+----+-----+----------+-------+-----------+--------+----------+----------+
|2015-01-01|20150101|2015|    1|   January|      1|          5|Thursday|     false|   2015-01|
|2015-01-02|20150102|2015|    1|   January|      1|          6|  Friday|     false|   2015-01|
|2015-01-03|20150103|2015|    1|   January|      1|          7|Saturday|      true|   2015-01|
+----------+--------+----+-----+----------+-------+-----------+--------+----------+----------+
only showing top 3 rows

Row count assertion passed ✅


In [3]:
# ============================================================
# CELL 3 — dim_pollutant (Static Reference Table)
# Type 0 SCD — never changes
# 5 rows: pm25, pm10, no2, co, o3
#
# Why a separate dimension for pollutants?
#   - WHO guideline values are reference data, not facts
#   - Centralizes threshold logic — one place to update
#   - Enables Power BI measures like:
#     "% readings exceeding WHO guideline"
#   - Avoids repeating guideline values in every fact row
#
# Interview note: "dim_pollutant is a Type 0 dimension —
# WHO guidelines change rarely, and when they do we do a
# full overwrite. No history needed here."
# ============================================================

pollutant_data = [
    # (pollutant_sk, code,   display_name, unit,    who_guideline, description)
    (1, "pm25", "PM2.5", "µg/m³",  15.0,
     "Fine particulate matter <2.5µm — penetrates deep into lungs and bloodstream"),
    (2, "pm10", "PM10",  "µg/m³",  45.0,
     "Coarse particulate matter <10µm — affects upper respiratory tract"),
    (3, "no2",  "NO2",   "µg/m³",  10.0,
     "Nitrogen dioxide — primary source: traffic and industrial combustion"),
    (4, "co",   "CO",    "µg/m³",  4000.0,
     "Carbon monoxide — colourless toxic gas from incomplete combustion"),
    (5, "o3",   "O3",    "µg/m³",  60.0,
     "Ground-level ozone — secondary pollutant formed by sunlight + NOx + VOCs")
]

schema_pollutant = StructType([
    # Integer surrogate key — faster joins than string codes
    StructField("pollutant_sk",        IntegerType()),
    # Natural key — matches parameter values in fact_readings
    StructField("pollutant_code",      StringType()),
    # Display name for Power BI labels
    StructField("pollutant_name",      StringType()),
    # Standard measurement unit per WHO guidelines
    StructField("standard_unit",       StringType()),
    # WHO 2021 annual mean guideline value
    # Used in fact table to compute exceeds_who_guideline flag
    StructField("who_guideline_value", DoubleType()),
    # Human-readable description for report tooltips
    StructField("description",         StringType())
])

df_pollutant = spark.createDataFrame(pollutant_data, schema=schema_pollutant)

# Write dim_pollutant
# mode=overwrite: safe — same 5 rows every run
# No partitioning: 5 rows, single file, always broadcast in joins
df_pollutant.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dim_pollutant")

pollutant_count = df_pollutant.count()
print(f"dim_pollutant: {pollutant_count} rows ✅")
print(f"\nPollutant reference table:")
df_pollutant.select(
    "pollutant_sk", "pollutant_code",
    "pollutant_name", "who_guideline_value", "standard_unit"
).show(truncate=False)

# Assertion: exactly 5 pollutants
assert pollutant_count == 5, f"Expected 5 rows, got {pollutant_count}"
print(f"Row count assertion passed ✅")

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 7, Finished, Available, Finished, False)

dim_pollutant: 5 rows ✅

Pollutant reference table:
+------------+--------------+--------------+-------------------+-------------+
|pollutant_sk|pollutant_code|pollutant_name|who_guideline_value|standard_unit|
+------------+--------------+--------------+-------------------+-------------+
|1           |pm25          |PM2.5         |15.0               |µg/m³        |
|2           |pm10          |PM10          |45.0               |µg/m³        |
|3           |no2           |NO2           |10.0               |µg/m³        |
|4           |co            |CO            |4000.0             |µg/m³        |
|5           |o3            |O3            |60.0               |µg/m³        |
+------------+--------------+--------------+-------------------+-------------+

Row count assertion passed ✅


In [4]:
# ============================================================
# CELL 4 — dim_country (SCD Type 1)
# Source: distinct countries from silver_readings
# SCD1: overwrite current values — no history kept
#
# Why Type 1 for countries?
#   - Country metadata (GDP, health spend) changes annually
#   - We don't need to track "what was India's GDP in 2019"
#   - Analysts always want the CURRENT enrichment values
#   - Historical GDP trends come from World Bank directly
#
# World Bank columns (gdp_per_capita, health_exp_pct, population)
# are NULL now — populated in Phase 2 via Dataflow Gen2
# World Bank API ingestion
#
# Interview note: "dim_country is SCD Type 1 — we overwrite
# because country-level economic indicators are reference data.
# The World Bank API gives us current values annually.
# We don't need point-in-time country GDP for air quality analysis."
# ============================================================

# --- Extract distinct countries from Silver ---
# Silver is our source of truth for which countries exist
# in the dataset — no hardcoding required
df_countries_raw = df_silver \
    .select("country_code", "country_name") \
    .distinct() \
    .filter(F.col("country_code") != "")  # drop blank country codes

# --- Generate surrogate key ---
# CRC32 hash of country_code → deterministic integer SK
# Same country_code always produces same SK across runs
# No sequence generator needed — avoids distributed counter issues
df_countries = df_countries_raw \
    .withColumn(
        "country_sk",
        F.crc32(F.col("country_code")).cast(IntegerType())
    ) \
    .withColumn(
        # Continent mapping — used for RLS in Power BI
        # Regional managers see only their continent's data
        # Extend this list as new countries appear in the feed
        "continent",
        F.when(F.col("country_code").isin(
            "IN", "CN", "TH", "MN", "JP", "KR", "SG",
            "MY", "ID", "PH", "VN", "BD", "PK", "NP", "LK"
        ), "Asia")
        .when(F.col("country_code").isin(
            "NL", "GB", "DE", "FR", "IT", "ES", "PL",
            "SE", "NO", "DK", "FI", "BE", "AT", "CH", "PT"
        ), "Europe")
        .when(F.col("country_code").isin(
            "US", "CA", "MX"
        ), "North America")
        .when(F.col("country_code").isin(
            "CL", "BR", "AR", "CO", "PE", "VE", "EC", "BO", "PY", "UY"
        ), "South America")
        .when(F.col("country_code").isin(
            "GH", "NG", "ZA", "KE", "ET", "EG", "MA", "TZ", "UG", "SN"
        ), "Africa")
        .when(F.col("country_code").isin(
            "AU", "NZ", "PG", "FJ"
        ), "Oceania")
        .otherwise("Other")
    ) \
    .withColumn(
        # GDP per capita USD — populated via World Bank API (Phase 2)
        # NULL placeholder for now
        "gdp_per_capita",
        F.lit(None).cast(DoubleType())
    ) \
    .withColumn(
        # Health expenditure as % of GDP — World Bank indicator
        # SH.XPD.CHEX.GD.ZS
        "health_exp_pct",
        F.lit(None).cast(DoubleType())
    ) \
    .withColumn(
        # Total population — World Bank indicator SP.POP.TOTL
        "population",
        F.lit(None).cast(LongType())
    ) \
    .withColumn(
        # Timestamp of last SCD1 update
        # Used to track when enrichment was last refreshed
        "scd_updated_ts",
        F.current_timestamp()
    )

# --- SCD Type 1 Write ---
# mode=overwrite: replaces entire table each run
# This IS correct for Type 1 — no history to preserve
# When World Bank data arrives, new columns merge via
# overwriteSchema=true
df_countries.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dim_country")

country_count = df_countries.count()
print(f"dim_country: {country_count} rows ✅")
print(f"\nCountry dimension:")
df_countries.select(
    "country_sk", "country_code",
    "country_name", "continent"
).show(truncate=False)

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 8, Finished, Available, Finished, False)

dim_country: 9 rows ✅

Country dimension:
+-----------+------------+--------------+-------------+
|country_sk |country_code|country_name  |continent    |
+-----------+------------+--------------+-------------+
|-437845118 |CL          |Chile         |South America|
|1346993615 |NL          |Netherlands   |Europe       |
|1714678657 |GB          |United Kingdom|Europe       |
|1954003872 |US          |United States |North America|
|-1788195040|MN          |Mongolia      |Asia         |
|-251229660 |IN          |India         |Asia         |
|-418823411 |TH          |Thailand      |Asia         |
|199844526  |CN          |China         |Asia         |
|-2079833584|PL          |Poland        |Europe       |
+-----------+------------+--------------+-------------+



In [5]:
# ============================================================
# CELL 5 — dim_station (SCD Type 2)
# Source: distinct stations from silver_readings
# Strategy: Delta MERGE — track station attribute changes
#
# Why Type 2 for stations?
#   - Station names and city assignments change over time
#     (e.g. a station is renamed or reassigned to a district)
#   - Analysts need point-in-time accuracy:
#     "What was this station called in 2024?"
#   - Compliance: regulatory reporting requires knowing
#     exactly which station configuration produced a reading
#
# SCD2 Implementation:
#   - active_flag = True → current version of the station
#   - active_flag = False → expired historical version
#   - effective_start → when this version became active
#   - effective_end → when this version was superseded
#                     (9999-12-31 = currently active)
#   - station_hash → MD5 of key attributes
#                    change in hash triggers new SCD2 row
#
# Delta MERGE steps:
#   Step 1: Match active rows where hash changed → expire them
#   Step 2: Insert new rows for changed + new stations
#
# Interview note: "We use Delta MERGE for SCD2 — one atomic
# operation handles expiry and insert together. The station_hash
# column detects any change in name or city without comparing
# every column individually. active_flag + effective_end=9999
# is the standard pattern for current-row identification."
# ============================================================

# --- Build station source from Silver ---
# Distinct stations with their current attributes
# .distinct() over all six columns did NOT give one row per station.
# OpenAQ returns city as locality-or-country name and its coordinate
# precision drifts between calls, so Silver holds several variants of the
# same station. Every variant hashed differently, survived the anti-join
# below, and was inserted as its own active_flag=true row — which is why
# dim_station accumulated multiple current rows per location_id and the
# fact join fanned out. Take the most recently observed variant instead.
_latest_variant = Window.partitionBy("location_id") \
                        .orderBy(F.col("reading_ts").desc())

df_stations_src = df_silver \
    .select(
        "location_id",
        "location_name",
        "city",
        "country_code",
        "latitude",
        "longitude",
        "reading_ts"
    ) \
    .withColumn("_rn", F.row_number().over(_latest_variant)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn", "reading_ts") \
    .withColumn(
        # MD5 hash of key attributes that we track for changes
        # If ANY of these change → SCD2 creates a new row
        # MD5 chosen over SHA256 for speed — collision risk
        # is acceptable for a station dimension
        "station_hash",
        F.md5(F.concat_ws("|",
            F.col("location_id").cast("string"),
            F.col("location_name"),
            F.col("city"),
            F.col("country_code")
        ))
    ) \
    .withColumn(
        # CRC32 surrogate key — deterministic from location_id
        # Same station always gets same SK across all runs
        "station_sk",
        F.crc32(F.col("location_id").cast("string")).cast(IntegerType())
    )

print(f"Source stations from Silver: {df_stations_src.count()}")

# --- Check if dim_station exists ---
# First run: CREATE fresh table with SCD2 columns
# Subsequent runs: MERGE to detect and handle changes
try:
    existing = spark.sql(
        f"SELECT COUNT(*) as cnt FROM {GOLD_DB}.dim_station"
    ).collect()[0]['cnt']
    table_exists = True
    print(f"dim_station exists — {existing} rows — running MERGE")
except Exception:
    table_exists = False
    print("dim_station does not exist — creating fresh")

if not table_exists:
    # --------------------------------------------------------
    # FIRST RUN — Create table with all SCD2 columns
    # All stations start as active with open effective_end
    # --------------------------------------------------------
    df_init = df_stations_src \
        .withColumn(
            # All rows active on first load
            "active_flag",
            F.lit(True)
        ) \
        .withColumn(
            # Version start = now (first time we saw this station)
            "effective_start",
            F.current_timestamp()
        ) \
        .withColumn(
            # Open-ended — 9999-12-31 means "currently active"
            # Standard SCD2 sentinel value
            "effective_end",
            F.lit("9999-12-31").cast(TimestampType())
        )

    df_init.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{GOLD_DB}.dim_station")

    init_count = df_init.count()
    print(f"dim_station created: {init_count} rows ✅")

else:
    # --------------------------------------------------------
    # SUBSEQUENT RUNS — SCD Type 2 MERGE
    #
    # Step 1: EXPIRE changed rows
    #   Match: station_id matches + active_flag=true + hash changed
    #   Action: set active_flag=False, effective_end=now
    #   Effect: old version is closed off with a timestamp
    #
    # Step 2: INSERT new/changed rows
    #   All source stations get inserted as new active rows
    #   Delta handles dedup — existing unchanged rows are
    #   NOT re-inserted (we filter in the source)
    # --------------------------------------------------------

    # Step 1: Expire changed rows
    print("Step 1: Expiring changed station rows...")
    DeltaTable.forName(spark, f"{GOLD_DB}.dim_station") \
        .alias("target") \
        .merge(
            df_stations_src.alias("source"),
            # Match condition: same station, currently active,
            # but attributes have changed (hash mismatch)
            """target.location_id = source.location_id
               AND target.active_flag = true
               AND target.station_hash <> source.station_hash"""
        ) \
        .whenMatchedUpdate(set={
            # Close off the old version
            "active_flag":   F.lit(False),
            "effective_end": F.current_timestamp()
        }) \
        .execute()
    print("  Changed rows expired ✅")

    # Step 2: Insert new active rows for changed + new stations
    # Only insert stations not already active with same hash
    print("Step 2: Inserting new/changed station rows...")
    df_existing_active = spark.sql(f"""
        SELECT location_id, station_hash
        FROM {GOLD_DB}.dim_station
        WHERE active_flag = true
    """)

    # Anti-join: source rows NOT in active target
    # These are either new stations or stations that just changed
    df_to_insert = df_stations_src \
        .join(
            df_existing_active,
            on=["location_id", "station_hash"],
            how="left_anti"  # keep source rows with no active match
        ) \
        .withColumn("active_flag",     F.lit(True)) \
        .withColumn("effective_start", F.current_timestamp()) \
        .withColumn("effective_end",
                    F.lit("9999-12-31").cast(TimestampType()))

    insert_count = df_to_insert.count()
    if insert_count > 0:
        df_to_insert.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(f"{GOLD_DB}.dim_station")
        print(f"  Inserted {insert_count} new/changed rows ✅")
    else:
        print("  No new or changed stations — nothing to insert ✅")

# --- Final state ---
final_count = spark.sql(
    f"SELECT COUNT(*) as cnt FROM {GOLD_DB}.dim_station"
).collect()[0]['cnt']
active_count = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {GOLD_DB}.dim_station
    WHERE active_flag = true
""").collect()[0]['cnt']

print(f"\ndim_station final state:")
print(f"  Total rows (all versions) : {final_count}")
print(f"  Active rows (current)     : {active_count}")
print(f"  Historical rows (expired) : {final_count - active_count}")

# --- Guard: exactly one active row per station ---
# A duplicate active row silently multiplies fact rows in the next cell,
# so stop here rather than let a fanned-out fact table reach Gold.
dupe_active = spark.sql(f"""
    SELECT location_id, COUNT(*) AS active_rows
    FROM {GOLD_DB}.dim_station
    WHERE active_flag = true
    GROUP BY location_id
    HAVING COUNT(*) > 1
""")
dupe_n = dupe_active.count()
if dupe_n > 0:
    print(f"\n❌ {dupe_n} station(s) have more than one active row:")
    dupe_active.show(truncate=False)
    raise AssertionError(
        f"dim_station SCD2 violated: {dupe_n} location_id(s) with multiple "
        "active rows. The fact join would fan out. Expire the stale versions "
        "before continuing."
    )
print("SCD2 check: one active row per station ✅")

print(f"\nSample active stations:")
spark.sql(f"""
    SELECT station_sk, location_id, location_name,
           city, country_code, active_flag,
           effective_start, effective_end
    FROM {GOLD_DB}.dim_station
    WHERE active_flag = true
    LIMIT 5
""").show(truncate=False)

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 9, Finished, Available, Finished, False)

Source stations from Silver: 121
dim_station does not exist — creating fresh
dim_station created: 121 rows ✅

dim_station final state:
  Total rows (all versions) : 121
  Active rows (current)     : 121
  Historical rows (expired) : 0

Sample active stations:
+-----------+-----------+--------------------------------------------+--------------------------+------------+-----------+--------------------------+-------------------+
|station_sk |location_id|location_name                               |city                      |country_code|active_flag|effective_start           |effective_end      |
+-----------+-----------+--------------------------------------------+--------------------------+------------+-----------+--------------------------+-------------------+
|1053240650 |149        |Ealing Horn Lane                            |London                    |GB          |true       |2026-08-08 16:58:32.954164|9999-12-31 00:00:00|
|-1218248294|156        |London Teddington                  

In [6]:
# ============================================================
# CELL 6 — fact_readings (Central Fact Table)
# Source: silver_readings joined to all 4 dimensions
# Grain: one row per station + pollutant + reading_timestamp
#
# Join strategy:
#   dim_country  → broadcast join (9 rows — always in memory)
#   dim_pollutant → broadcast join (5 rows — always in memory)
#   dim_station  → regular join (121 rows — small enough to broadcast)
#   dim_date     → not joined here — Power BI handles via date_key
#
# Why broadcast for small dims?
#   Eliminates shuffle entirely — no data movement across executors
#   For a 344-row fact table joining 9-row dim_country:
#   broadcast avoids 344 rows being reshuffled across the cluster
#   At scale (billions of rows), this saves minutes of shuffle time
#
# V-Order:
#   Fabric-specific Parquet write optimization
#   Sorts data within each row group by value distribution
#   Direct Lake reads V-Ordered files without loading into memory
#   MANDATORY for Direct Lake semantic models in Power BI
#   Without V-Order, Direct Lake falls back to DirectQuery mode
#
# Z-Order:
#   Delta file-level clustering on location_id + reading_ts
#   Co-locates data for the most common query pattern:
#   "Show me station X readings between date A and date B"
#   Reduces files scanned from N → 1-2 for filtered queries
#
# Interview note: "fact_readings is written with V-Order because
# our Power BI semantic model uses Direct Lake mode — V-Order
# is what makes Direct Lake fast. Without it, Power BI would
# fall back to DirectQuery which defeats the purpose. Z-Order
# on location_id + reading_ts matches our primary filter pattern
# in the RTI dashboard and Power BI reports."
# ============================================================

# --- Load dimensions for joins ---
# These are small tables — broadcast to all executors
# No shuffle needed — each executor has full copy in memory
df_country = spark.read.format("delta") \
    .table(f"{GOLD_DB}.dim_country") \
    .select("country_code", "country_sk", "continent")

df_pollutant = spark.read.format("delta") \
    .table(f"{GOLD_DB}.dim_pollutant") \
    .select(
        F.col("pollutant_code"),
        F.col("pollutant_sk"),
        F.col("who_guideline_value")
    )

df_station = spark.read.format("delta") \
    .table(f"{GOLD_DB}.dim_station") \
    .filter(F.col("active_flag") == True) \
    .select("location_id", "station_sk")

print(f"Dimensions loaded:")
print(f"  dim_country  : {df_country.count()} rows")
print(f"  dim_pollutant: {df_pollutant.count()} rows")
print(f"  dim_station  : {df_station.count()} rows (active only)")

# --- Build fact table ---
# Join Silver to dimensions
# broadcast() hint forces Spark to broadcast the dimension
# even if AQE would choose a different strategy
df_fact = df_silver \
    .join(
        # Broadcast dim_country — 9 rows, ~1KB
        F.broadcast(df_country),
        on="country_code",
        how="left"   # left join: keep all fact rows even if no country match
    ) \
    .join(
        # Broadcast dim_pollutant — 5 rows, <1KB
        # Join on parameter name = pollutant_code
        F.broadcast(df_pollutant),
        df_silver["parameter"] == df_pollutant["pollutant_code"],
        how="left"
    ) \
    .join(
        # dim_station — 121 rows, also small enough to broadcast
        F.broadcast(df_station),
        on="location_id",
        how="left"
    ) \
    .withColumn(
        # WHO exceedance flag
        # True when reading exceeds WHO annual mean guideline
        # Key metric for Data Activator alerts and PBI reports
        # NULL when pollutant has no guideline match
        "exceeds_who_guideline",
        F.when(
            F.col("who_guideline_value").isNotNull(),
            F.col("value") > F.col("who_guideline_value")
        ).otherwise(F.lit(False))
    ) \
    .withColumn(
        # Integer date key for joining to dim_date in Power BI
        # e.g. 2026-08-08 → 20260808
        "date_key",
        F.date_format("reading_date", "yyyyMMdd").cast(IntegerType())
    ) \
    .select(
        # --- Surrogate keys (for Power BI relationships) ---
        "station_sk",
        "country_sk",
        "pollutant_sk",
        "date_key",

        # --- Degenerate dimension (natural key kept in fact) ---
        "location_id",

        # --- Measures ---
        "parameter",
        "value",
        "unit",
        "aqi_category",
        "exceeds_who_guideline",

        # --- Time attributes ---
        "reading_ts",
        "reading_date",
        "reading_hour",
        "year_month",
        "is_recent",

        # --- Spatial attributes ---
        "latitude",
        "longitude",

        # --- Continent (denormalized from dim_country) ---
        # Kept in fact for RLS filter performance
        "continent",

        # --- Lineage ---
        "source_system",
        "ingestion_ts"
    )

fact_count = df_fact.count()
print(f"\nFact rows before write: {fact_count}")

# --- Write fact_readings with V-Order ---
# V-Order is enabled by default in Fabric Spark runtime
# It applies Microsoft's proprietary Parquet optimization
# that sorts values within row groups for Direct Lake efficiency
print("Writing fact_readings with V-Order...")
df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .partitionBy("year_month") \
    .saveAsTable(f"{GOLD_DB}.fact_readings")

print(f"fact_readings written: {fact_count} rows ✅")

# --- Apply Z-Order ---
# Z-Order clusters Delta files on the specified columns
# Co-locates rows with similar location_id + reading_ts values
# Dramatically reduces files scanned for filtered queries:
# "WHERE location_id = 173 AND reading_ts >= '2026-01-01'"
# Without Z-Order: scan all files
# With Z-Order: scan 1-2 files maximum
print("\nApplying Z-Order on location_id + reading_ts...")
spark.sql(f"""
    OPTIMIZE {GOLD_DB}.fact_readings
    ZORDER BY (location_id, reading_ts)
""")
print("Z-Order applied ✅")

# --- Show sample ---
print("\nSample fact rows:")
df_fact.select(
    "station_sk", "country_sk", "pollutant_sk",
    "parameter", "value", "aqi_category",
    "exceeds_who_guideline", "continent", "reading_ts"
).show(5, truncate=False)

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 10, Finished, Available, Finished, False)

Dimensions loaded:
  dim_country  : 9 rows
  dim_pollutant: 5 rows
  dim_station  : 121 rows (active only)

Fact rows before write: 344
Writing fact_readings with V-Order...
fact_readings written: 344 rows ✅

Applying Z-Order on location_id + reading_ts...
Z-Order applied ✅

Sample fact rows:
+----------+----------+------------+---------+-----+------------+---------------------+---------+-------------------+
|station_sk|country_sk|pollutant_sk|parameter|value|aqi_category|exceeds_who_guideline|continent|reading_ts         |
+----------+----------+------------+---------+-----+------------+---------------------+---------+-------------------+
|-955648747|1714678657|2           |pm10     |8.0  |N/A         |false                |Europe   |2022-05-11 23:00:00|
|-955648747|1714678657|1           |pm25     |5.0  |Good        |false                |Europe   |2022-05-12 00:00:00|
|1060745282|-251229660|4           |co       |1.65 |N/A         |false                |Asia     |2025-02-18 19:30:00

In [8]:
# ============================================================
# CELL 7 — Gold Layer Validation
# Checks: row counts, null rates, WHO exceedances,
#         join integrity, Delta table details
#
# This is our "Definition of Done" check for Gold layer
# If any assertion fails, the pipeline should alert
# and NOT update the downstream semantic model
# ============================================================

print("=" * 55)
print("GOLD LAYER VALIDATION REPORT")
print("=" * 55)

# --- Table Row Counts ---
# All 5 tables must exist and have expected row counts
print("\n1. Table row counts:")
tables = {
    "dim_date"     : 5844,   # fixed — 2015 to 2030
    "dim_pollutant": 5,      # fixed — 5 pollutants
    "dim_country"  : None,   # variable — depends on feed
    "dim_station"  : None,   # variable — depends on feed
    "fact_readings": None    # variable — depends on feed
}
for table, expected in tables.items():
    cnt = spark.sql(
        f"SELECT COUNT(*) as cnt FROM {GOLD_DB}.{table}"
    ).collect()[0]['cnt']
    status = "✅" if (expected is None or cnt == expected) else "❌"
    print(f"  {status} {table}: {cnt} rows"
          + (f" (expected {expected})" if expected else ""))

# --- Null Check on fact_readings ---
# Surrogate keys must never be null — broken joins produce nulls
print("\n2. Null check on fact_readings surrogate keys:")
spark.sql(f"""
    SELECT
        SUM(CASE WHEN station_sk  IS NULL THEN 1 ELSE 0 END) as null_station_sk,
        SUM(CASE WHEN country_sk  IS NULL THEN 1 ELSE 0 END) as null_country_sk,
        SUM(CASE WHEN pollutant_sk IS NULL THEN 1 ELSE 0 END) as null_pollutant_sk,
        SUM(CASE WHEN date_key    IS NULL THEN 1 ELSE 0 END) as null_date_key,
        SUM(CASE WHEN value       IS NULL THEN 1 ELSE 0 END) as null_value
    FROM {GOLD_DB}.fact_readings
""").show()

# --- WHO Exceedance Summary ---
# Core business metric — which countries exceed WHO guidelines
print("\n3. WHO Guideline Exceedances by country + pollutant:")
spark.sql(f"""
    SELECT
        c.country_name,
        c.continent,
        f.parameter,
        COUNT(*)                                          AS total_readings,
        SUM(CASE WHEN f.exceeds_who_guideline
                 THEN 1 ELSE 0 END)                      AS exceedances,
        ROUND(
            SUM(CASE WHEN f.exceeds_who_guideline
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(*), 1
        )                                                AS exceedance_pct,
        ROUND(AVG(f.value), 2)                           AS avg_value
    FROM {GOLD_DB}.fact_readings f
    JOIN {GOLD_DB}.dim_country c
      ON f.country_sk = c.country_sk
    WHERE f.parameter IN ('pm25', 'no2')
    GROUP BY c.country_name, c.continent, f.parameter
    ORDER BY exceedances DESC
    LIMIT 10
""").show(truncate=False)

# --- AQI Distribution ---
print("\n4. AQI category distribution (PM2.5 only):")
spark.sql(f"""
    SELECT
        aqi_category,
        COUNT(*) as readings,
        ROUND(COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (), 1) as pct
    FROM {GOLD_DB}.fact_readings
    WHERE parameter = 'pm25'
    GROUP BY aqi_category
    ORDER BY readings DESC
""").show()

# --- SCD2 Integrity Check ---
print("\n5. dim_station SCD2 integrity check:")
spark.sql(f"""
    SELECT
        COUNT(*)                                              AS total_rows,
        SUM(CASE WHEN active_flag = true  THEN 1 ELSE 0 END) AS active_rows,
        SUM(CASE WHEN active_flag = false THEN 1 ELSE 0 END) AS expired_rows,
        SUM(CASE WHEN active_flag = true
                 AND effective_end = '9999-12-31'
                 THEN 1 ELSE 0 END)                          AS valid_active_rows,
        COUNT(DISTINCT location_id)                          AS distinct_stations
    FROM {GOLD_DB}.dim_station
""").show()

# active_rows must equal distinct_stations — any excess is a station
# carrying more than one current version, which fans out the fact join.
_scd2 = spark.sql(f"""
    SELECT COUNT(*) AS offenders FROM (
        SELECT location_id
        FROM {GOLD_DB}.dim_station
        WHERE active_flag = true
        GROUP BY location_id
        HAVING COUNT(*) > 1
    )
""").collect()[0]["offenders"]
print(f"  Stations with >1 active row: {_scd2} "
      + ("✅" if _scd2 == 0 else "❌"))

# --- Delta Table Details ---
print("\n6. Delta table details (fact_readings):")
spark.sql(f"""
    DESCRIBE DETAIL {GOLD_DB}.fact_readings
""").select(
    "format",
    "numFiles",
    "sizeInBytes",
    "location"
).show(truncate=False)

# --- Final Assertions ---
print("\n7. Running assertions...")
fact_count = spark.sql(
    f"SELECT COUNT(*) as cnt FROM {GOLD_DB}.fact_readings"
).collect()[0]['cnt']
assert fact_count > 0, \
    "❌ FAILED: fact_readings is empty"

null_sks = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {GOLD_DB}.fact_readings
    WHERE station_sk IS NULL
    OR country_sk IS NULL
    OR pollutant_sk IS NULL
""").collect()[0]['cnt']
assert null_sks == 0, \
    f"❌ FAILED: {null_sks} rows have null surrogate keys"

active_stations = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {GOLD_DB}.dim_station
    WHERE active_flag = true
    AND effective_end = '9999-12-31 00:00:00'
""").collect()[0]['cnt']
assert active_stations > 0, \
    "❌ FAILED: No active stations in dim_station"

print("  All assertions passed ✅")
print("\n" + "=" * 55)
print("✅ GOLD LAYER COMPLETE")
print("   Tables: dim_date, dim_pollutant, dim_country,")
print("           dim_station (SCD2), fact_readings")
print("   Next: Configure Direct Lake semantic model")
print("=" * 55)

StatementMeta(, f73e0ac3-1d10-42cf-b9bb-7b4226c45c61, 12, Finished, Available, Finished, False)

GOLD LAYER VALIDATION REPORT

1. Table row counts:
  ✅ dim_date: 5844 rows (expected 5844)
  ✅ dim_pollutant: 5 rows (expected 5)
  ✅ dim_country: 9 rows
  ✅ dim_station: 121 rows
  ✅ fact_readings: 344 rows

2. Null check on fact_readings surrogate keys:
+---------------+---------------+-----------------+-------------+----------+
|null_station_sk|null_country_sk|null_pollutant_sk|null_date_key|null_value|
+---------------+---------------+-----------------+-------------+----------+
|              0|              0|                0|            0|         0|
+---------------+---------------+-----------------+-------------+----------+


3. WHO Guideline Exceedances by country + pollutant:
+--------------+-------------+---------+--------------+-----------+--------------+---------+
|country_name  |continent    |parameter|total_readings|exceedances|exceedance_pct|avg_value|
+--------------+-------------+---------+--------------+-----------+--------------+---------+
|United Kingdom|Europe   